# Australian Petroleum Statistics

Data from the Department of Climate Change, Energy, the Environment and Water (DCCEEW).

Source: https://www.energy.gov.au/publications/australian-petroleum-statistics

## Set-up

In [ ]:
# system imports
from pathlib import Path
from urllib.parse import unquote

# analytic imports
import pandas as pd
import requests
from bs4 import BeautifulSoup
from mgplot import (
    line_plot_finalise,
    bar_plot_finalise,
    set_chart_dir,
    clear_chart_dir,
)

In [ ]:
# plotting set-up
CHART_DIR = "./CHARTS/DCCEEW-Petroleum/"
set_chart_dir(CHART_DIR)
clear_chart_dir()
SHOW = False

SOURCE = "DCCEEW, Petroleum Statistics"
LFOOTER = "Australia. "

## Data Acquisition

In [ ]:
# Find and download the latest spreadsheet from the publication page
from datetime import date

BASE_URL = "https://www.energy.gov.au"
PAGE_TEMPLATE = BASE_URL + "/publications/australian-petroleum-statistics-{year}"

# Try current year first, fall back to previous year
# (new year's page may not exist until ~March)
year = date.today().year
for try_year in (year, year - 1):
    page_url = PAGE_TEMPLATE.format(year=try_year)
    page = requests.get(page_url, timeout=60)
    if page.status_code != 200:
        continue
    soup = BeautifulSoup(page.text, "html.parser")
    xlsx_links = soup.find_all("a", href=lambda h: h and h.endswith(".xlsx"))
    if xlsx_links:
        break
else:
    raise ValueError("No spreadsheet found for current or previous year")

# Last link on the page is the most recent month
href = xlsx_links[-1]["href"]
url = href if href.startswith("http") else BASE_URL + href

CACHE_DIR = "./DCCEEW_CACHE/"
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
CACHE_FILE = CACHE_DIR + unquote(url.split("/")[-1])

if not Path(CACHE_FILE).exists():
    print(f"Downloading: {unquote(url.split('/')[-1])}")
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    with open(CACHE_FILE, "wb") as f:
        f.write(response.content)
    print(f"Saved to {CACHE_FILE}")
else:
    print(f"Using cached file: {CACHE_FILE}")

In [ ]:
def read_monthly_sheet(sheet_name: str) -> pd.DataFrame:
    """Read a monthly sheet from the petroleum statistics spreadsheet."""
    df = pd.read_excel(CACHE_FILE, sheet_name=sheet_name)
    df = df.dropna(how="all", axis=0)
    df = df.set_index("Month")
    df.index = pd.PeriodIndex(df.index, freq="M")
    # Replace 'n.a.' with NaN
    df = df.replace("n.a.", pd.NA)
    return df


# Load key sheets
production = read_monthly_sheet("Petroleum production")
sales = read_monthly_sheet("Sales of products")
imports_vol = read_monthly_sheet("Imports volume")
exports_vol = read_monthly_sheet("Exports volume")
refinery = read_monthly_sheet("Refinery production")
stocks = read_monthly_sheet("Stock volume by product")
consumption_cover = read_monthly_sheet("Consumption cover")
iea_cover = read_monthly_sheet("IEA days net import cover")

# Fuel prices are quarterly with a different format
fuel_prices = pd.read_excel(CACHE_FILE, sheet_name="Australian fuel prices")
fuel_prices = fuel_prices.dropna(how="all", axis=0)
fuel_prices.index = pd.PeriodIndex(
    fuel_prices["Year"].astype(str) + fuel_prices["Quarter"], freq="Q"
)
fuel_prices = fuel_prices.drop(columns=["Year", "Quarter"])

print(f"Data from {production.index[0]} to {production.index[-1]}")
print(f"Fuel prices from {fuel_prices.index[0]} to {fuel_prices.index[-1]}")

## IEA Days of Net Import Cover

In [ ]:
IEA_LFOOTER = (
    LFOOTER
    + "Monthly. Crude-oil-equivalent aggregate. Cover based on prior year ave daily consumption, updated each April."
)
DNI_LFOOTER = LFOOTER + "Average daily net imports for previous calendar year. Updated each April."

line_plot_finalise(
    iea_cover["IEA days of net import coverage"],
    title="IEA Days of Net Import Coverage",
    ylabel="Days",
    rfooter=SOURCE,
    lfooter=IEA_LFOOTER,
    show=SHOW,
)

line_plot_finalise(
    iea_cover["Daily Net Imports (kT/day)"],
    title="Daily Net Imports",
    ylabel="kT / day",
    rfooter=SOURCE,
    lfooter=DNI_LFOOTER,
    show=SHOW,
)

## Consumption Cover by Product

In [ ]:
COVER_LFOOTER = LFOOTER + "Monthly. Stocks / rolling 12-month ave daily consumption."

# Key products on one chart
key_products = consumption_cover[
    [
        "Crude oil and refinery feedstocks (days)",
        "Automotive gasoline (days)",
        "Diesel oil (days)",
        "Aviation turbine fuel (days)",
    ]
].rename(columns=lambda x: x.replace(" (days)", ""))

line_plot_finalise(
    key_products,
    title="Consumption Cover by Product",
    ylabel="Days of supply",
    legend={"loc": "best", "fontsize": "small"},
    rfooter=SOURCE,
    lfooter=COVER_LFOOTER,
    show=SHOW,
)

# Individual product charts
for col in key_products.columns:
    line_plot_finalise(
        key_products[col],
        title=f"Consumption Cover: {col}",
        ylabel="Days of supply",
        rfooter=SOURCE,
        lfooter=COVER_LFOOTER,
        show=SHOW,
    )

## Structural Shift: Production, Refining and Import Dependence

In [ ]:
MONTHLY_LFOOTER = LFOOTER + "Monthly."

# Overall trade balance and domestic consumption
trade_consumption = pd.DataFrame({
    "Total imports": imports_vol["Total oil imports (ML)"],
    "Total exports": exports_vol["Total oil exports (inc. ships' and aircraft stores) (ML)"],
    "Domestic sales": sales["Total (ML)"],
})
line_plot_finalise(
    trade_consumption,
    title="Petroleum: Imports, Exports and Domestic Sales",
    ylabel="Megalitres / Month",
    legend={"loc": "best", "fontsize": "small"},
    rfooter=SOURCE,
    lfooter=MONTHLY_LFOOTER + " Exports exclude LNG. Exports are mainly crude oil & condensate.",
    show=SHOW,
)

# Domestic crude oil & condensate production
line_plot_finalise(
    production["Crude oil & condensate (ML)"],
    title="Domestic Crude Oil & Condensate Production",
    ylabel="Megalitres / Month",
    rfooter=SOURCE,
    lfooter=MONTHLY_LFOOTER,
    show=SHOW,
)

# Refinery total input - shows refinery closures
line_plot_finalise(
    refinery["Total input (ML)"],
    title="Refinery Total Input",
    ylabel="Megalitres / Month",
    rfooter=SOURCE,
    lfooter=MONTHLY_LFOOTER,
    show=SHOW,
)

# Indigenous share of refinery input
line_plot_finalise(
    refinery["Percentage indigenous: Total input (%)"],
    title="Refinery Input: Indigenous Crude Share",
    ylabel="Per cent",
    rfooter=SOURCE,
    lfooter=MONTHLY_LFOOTER,
    show=SHOW,
)

# Domestic crude vs imported feedstock vs imported refined products
imports_vs_production = pd.DataFrame({
    "Domestic crude oil & condensate": production["Crude oil & condensate (ML)"],
    "Imported crude & feedstock": imports_vol["Crude oil & other refinery feedstocks (ML)"],
    "Imported refined products": imports_vol["Total refined petroleum products (ML)"],
})
line_plot_finalise(
    imports_vs_production,
    title="Domestic Production vs Imports",
    ylabel="Megalitres / Month",
    legend={"loc": "best", "fontsize": "small"},
    rfooter=SOURCE,
    lfooter=MONTHLY_LFOOTER,
    show=SHOW,
)

# Import share: refined products as % of total oil imports
import_share = (
    imports_vol["Total refined petroleum products (ML)"]
    / imports_vol["Total oil imports (ML)"]
    * 100
)
line_plot_finalise(
    import_share,
    title="Refined Product Imports as Share of Total Oil Imports",
    ylabel="Per cent",
    rfooter=SOURCE,
    lfooter=MONTHLY_LFOOTER,
    show=SHOW,
)

## LNG Exports

In [ ]:
# LNG exports from production sheet (Mm3) and exports sheet (ML)
lng_production = pd.to_numeric(
    production["LNG exports (Mm3)"], errors="coerce"
).dropna()
line_plot_finalise(
    lng_production,
    title="LNG Exports",
    ylabel="Mm³ / Month",
    rfooter=SOURCE,
    lfooter=MONTHLY_LFOOTER,
    show=SHOW,
)

lng_ml = pd.to_numeric(
    exports_vol["LNG (ML)"], errors="coerce"
).dropna()
line_plot_finalise(
    lng_ml,
    title="LNG Exports (Volume)",
    ylabel="Megalitres / Month",
    rfooter=SOURCE,
    lfooter=MONTHLY_LFOOTER,
    show=SHOW,
)

## Australian Fuel Prices

In [ ]:
PRICE_LFOOTER = LFOOTER + "Quarterly average retail prices, sales-weighted by state."

price_cols = fuel_prices.rename(columns=lambda x: x.replace(" (cpl)", ""))

line_plot_finalise(
    price_cols,
    title="Australian Retail Fuel Prices",
    ylabel="Cents per litre",
    legend={"loc": "best", "fontsize": "small"},
    rfooter=SOURCE,
    lfooter=PRICE_LFOOTER,
    show=SHOW,
)

# ULP vs Diesel
line_plot_finalise(
    price_cols[["Regular unleaded petrol (91 RON)", "Automotive diesel"]],
    title="Petrol vs Diesel Prices",
    ylabel="Cents per litre",
    legend={"loc": "best", "fontsize": "small"},
    rfooter=SOURCE,
    lfooter=PRICE_LFOOTER,
    show=SHOW,
)

## Finished

In [ ]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda